# Лабораторная работа № 1
## Многомерная оптимизация сложной функции генетическим алгоритмом

**Дисциплина:** «Эволюционное программирование и генетические алгоритмы»

### Индивидуальный вариант

| Параметр | Значение |
|---|---|
| N (номер в группе) | 10 |
| Q (группа 507) | 0 |
| Y | 26 |
| V = 1 + ((17N + 7Q + Y) mod 20) | 1 + (196 mod 20) = **17** |
| d = 5 + (V mod 6) | 5 + 5 = **10** |

**Функция варианта 17 — функция различных степеней (Sum of Different Powers):**

$$ f(\mathbf{x}) = \sum_{i=1}^{d} |x_i|^{\,i+1}, \qquad \mathbf{x} \in D = [-1, 1]^{d},\ d = 10. $$

Теоретический глобальный минимум: $f(\mathbf{x}^*) = 0$ при $\mathbf{x}^* = (0, \dots, 0)$.

### Формальная постановка

Найти $\mathbf{x}^* = \arg\min_{\mathbf{x} \in D} f(\mathbf{x})$, где $D = \{\mathbf{x} \in \mathbb{R}^{10} : -1 \le x_i \le 1\}$.
Ограничения только box-типа (границы по каждой координате).

### Представление особи

* **Генотип** — вещественный вектор $\mathbf{x} \in \mathbb{R}^{10}$ (вещественное кодирование); при самоадаптивной мутации к нему добавляется
  стратегический параметр $\sigma > 0$ (масштаб мутации конкретной особи).
* **Фенотип** совпадает с генотипом (декодирование не требуется), $\sigma$ на фенотип не влияет.
* **Фитнес** — значение $f(\mathbf{x})$, которое минимизируется (меньше — лучше).

## 1. Реализация

Сначала определим все функции, которые понадобятся: целевую функцию со счётчиком вычислений, операторы ГА
(инициализацию, селекцию, кроссовер, мутацию, обработку границ), сам ГА, базовые методы для сравнения
(случайный поиск и локальный поиск), а также функции для серий запусков, статистики и графиков.
Из библиотек используются только `numpy` (численные операции, ГПСЧ), `pandas` (таблицы), `matplotlib` (графики)
и `scipy.stats` (статистический тест). Все эволюционные операторы реализованы вручную.

In [1]:
import time
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

D = 10                 # размерность варианта
LOW, HIGH = -1.0, 1.0  # границы области поиска
F_OPT = 0.0            # теоретический минимум

plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3})

### 1.1. Целевая функция

`sum_diff_powers` векторизована: принимает матрицу `(n, d)` и возвращает `n` значений.
Класс `Objective` считает число вычислений функции — бюджет всех методов задаётся именно в вычислениях $f$.

In [2]:
def sum_diff_powers(X):
    """f(x) = sum_i |x_i|^(i+1), i = 1..d. X: (n, d) или (d,)."""
    X = np.atleast_2d(X)
    powers = np.arange(2, X.shape[1] + 2)
    return np.sum(np.abs(X) ** powers, axis=1)


class Objective:
    """Обёртка над функцией со счётчиком вычислений."""

    def __init__(self, func, d, low, high):
        self.func, self.d, self.low, self.high = func, d, low, high
        self.evals = 0

    def __call__(self, X):
        X = np.atleast_2d(X)
        self.evals += X.shape[0]
        return self.func(X)

### 1.2. Инициализация и обработка границ

* Инициализация — равномерно в $D$, т.е. все стартовые особи допустимы.
* Выход за границы после мутации/кроссовера обрабатывается **отражением** от границы
  (точка $x < l$ переходит в $l + (l - x)$). В отличие от простого обрезания (`clip`), отражение не
  создаёт искусственного скопления особей на самой границе. Для сравнения реализован и `clip`.

In [3]:
def init_population(rng, n, d, low, high):
    return rng.uniform(low, high, size=(n, d))


def repair_bounds(X, low, high, mode="reflect"):
    if mode == "clip":
        return np.clip(X, low, high)
    width = high - low
    # отражение с периодом 2*width: корректно и для сильных выходов за границу
    Y = np.mod(X - low, 2 * width)
    Y = np.where(Y > width, 2 * width - Y, Y)
    return low + Y

### 1.3. Селекция родителей

Задача на минимизацию, поэтому все операторы работают с $f$ «наоборот»:

* **Турнирная** (размер турнира $k$): из $k$ случайных особей выбирается особь с наименьшим $f$.
  Селективное давление регулируется $k$.
* **Ранговая** (линейное ранжирование, давление $s \in [1, 2]$): вероятность выбора зависит только от ранга,
  а не от масштаба $f$ — это важно, т.к. значения $f$ в популяции отличаются на порядки.
* **Рулеточная с масштабированием окном**: $w_j = f_{\max} - f_j + \varepsilon$. Без масштабирования рулетка для
  задачи минимизации неприменима.

In [4]:
def select_tournament(rng, fit, n_select, k=3):
    idx = rng.integers(0, len(fit), size=(n_select, k))
    winners = np.argmin(fit[idx], axis=1)
    return idx[np.arange(n_select), winners]


def select_rank(rng, fit, n_select, s=1.7):
    n = len(fit)
    order = np.argsort(-fit)                 # от худшего к лучшему
    ranks = np.empty(n)
    ranks[order] = np.arange(n)              # худший -> 0, лучший -> n-1
    p = (2 - s) / n + 2 * ranks * (s - 1) / (n * (n - 1))
    return rng.choice(n, size=n_select, p=p / p.sum())


def select_roulette(rng, fit, n_select, eps=1e-12):
    w = fit.max() - fit + eps                # масштабирование окном
    return rng.choice(len(fit), size=n_select, p=w / w.sum())


SELECTIONS = {"tournament": select_tournament, "rank": select_rank, "roulette": select_roulette}

### 1.4. Кроссовер для вещественных векторов

* **SBX** (simulated binary crossover, параметр распределения $\eta$): потомки распределены около родителей,
  при большом $\eta$ — ближе к родителям. Основной оператор.
* **BLX-α**: потомок равномерно выбирается из отрезка $[\min - \alpha I, \max + \alpha I]$ по каждой координате.
* **Арифметический**: $c_1 = \lambda p_1 + (1-\lambda) p_2$, $c_2 = (1-\lambda) p_1 + \lambda p_2$.

Каждая функция принимает два массива родителей `(m, d)` и возвращает двух потомков той же формы.

In [5]:
def crossover_sbx(rng, P1, P2, eta=15.0):
    u = rng.random(P1.shape)
    beta = np.where(u <= 0.5, (2 * u) ** (1 / (eta + 1)), (1 / (2 * (1 - u))) ** (1 / (eta + 1)))
    C1 = 0.5 * ((1 + beta) * P1 + (1 - beta) * P2)
    C2 = 0.5 * ((1 - beta) * P1 + (1 + beta) * P2)
    # каждая координата меняется с вероятностью 0.5 (как в стандартном SBX)
    swap = rng.random(P1.shape) < 0.5
    return np.where(swap, C1, P1), np.where(swap, C2, P2)


def crossover_blx(rng, P1, P2, alpha=0.5):
    lo, hi = np.minimum(P1, P2), np.maximum(P1, P2)
    span = hi - lo
    a, b = lo - alpha * span, hi + alpha * span
    return rng.uniform(a, b), rng.uniform(a, b)


def crossover_arith(rng, P1, P2):
    lam = rng.random((P1.shape[0], 1))
    return lam * P1 + (1 - lam) * P2, (1 - lam) * P1 + lam * P2


CROSSOVERS = {"sbx": crossover_sbx, "blx": crossover_blx, "arith": crossover_arith}

### 1.5. Мутация

Гауссовская мутация: каждый ген с вероятностью $p_m$ получает добавку $\mathcal{N}(0, \sigma^2)$.

* **Фиксированный масштаб**: $\sigma = \sigma_0 \cdot (h - l)$ одинаков на всём протяжении поиска.
* **Самоадаптивный масштаб** (дополнительное задание): у каждой особи свой $\sigma$, который сначала мутирует
  логнормально $\sigma' = \sigma \cdot e^{\tau \mathcal{N}(0,1)}$, $\tau = 1/\sqrt{d}$, а затем используется
  для мутации генов. Потомок наследует среднее геометрическое $\sigma$ родителей. Удачные масштабы
  «выживают» вместе с удачными особями, поэтому $\sigma$ уменьшается по мере приближения к оптимуму —
  это позволяет достигать высокой точности, недоступной при фиксированном шаге.

In [6]:
def mutate_gaussian(rng, X, sigma, pm):
    """sigma: скаляр или массив (n,) — масштаб для каждой особи."""
    sigma = np.broadcast_to(np.asarray(sigma, dtype=float).reshape(-1, 1), (X.shape[0], 1))
    mask = rng.random(X.shape) < pm
    return X + mask * rng.normal(0.0, 1.0, X.shape) * sigma


def self_adapt_sigma(rng, sigma, tau, sigma_min=1e-12, sigma_max=1.0):
    return np.clip(sigma * np.exp(tau * rng.normal(size=sigma.shape)), sigma_min, sigma_max)

### 1.6. Генетический алгоритм

Схема — поколенческий ГА с элитизмом:

```
 ┌──────────────────────────────┐
 │ Инициализация P(0) ∈ D,      │
 │ оценка f, evals = μ          │
 └──────────────┬───────────────┘
                ▼
 ┌──────────────────────────────┐  да   ┌─────────────────────────┐
 │ evals + (μ - e) > бюджет ? ──┼──────▶│ вернуть лучшее решение  │
 └──────────────┬───────────────┘       └─────────────────────────┘
                │ нет
                ▼
 ┌──────────────────────────────┐
 │ Селекция μ - e родителей     │  турнир / ранг / рулетка
 └──────────────┬───────────────┘
                ▼
 ┌──────────────────────────────┐
 │ Кроссовер пар с вер. p_c     │  SBX / BLX-α / арифм.
 └──────────────┬───────────────┘
                ▼
 ┌──────────────────────────────┐
 │ (самоадаптация σ)            │
 │ Гауссовская мутация, p_m     │
 └──────────────┬───────────────┘
                ▼
 ┌──────────────────────────────┐
 │ Отражение от границ D        │
 │ Оценка f потомков            │
 └──────────────┬───────────────┘
                ▼
 ┌──────────────────────────────┐
 │ P(t+1) = e лучших из P(t)    │  элитизм
 │        + μ - e потомков      │
 └──────────────┬───────────────┘
                └──────▶ (к проверке бюджета)
```

Все параметры собраны в `GAConfig`. Критерий остановки — исчерпание бюджета вычислений $f$
(поколение запускается, только если на всех его потомков хватает бюджета).

In [7]:
@dataclass
class GAConfig:
    pop_size: int = 50            # μ — размер популяции
    max_evals: int = 20_000       # бюджет вычислений f
    selection: str = "tournament" # tournament | rank | roulette
    tournament_k: int = 3
    rank_s: float = 1.7
    crossover: str = "sbx"        # sbx | blx | arith
    pc: float = 0.9               # вероятность кроссовера для пары
    sbx_eta: float = 15.0
    blx_alpha: float = 0.5
    pm: float = 0.1               # вероятность мутации гена (= 1/d)
    sigma0: float = 0.1           # начальный масштаб мутации, доля ширины области
    adaptive: bool = True         # самоадаптация σ
    elite: int = 2                # число элитных особей
    bounds: str = "reflect"       # reflect | clip


def run_ga(cfg: GAConfig, seed: int, func=sum_diff_powers, d=D, low=LOW, high=HIGH):
    """Один запуск ГА. Возвращает лучшее решение и историю по поколениям."""
    rng = np.random.default_rng(seed)
    obj = Objective(func, d, low, high)
    mu, e = cfg.pop_size, cfg.elite
    n_off = mu - e
    n_pairs = (n_off + 1) // 2
    tau = 1 / np.sqrt(d)
    select = SELECTIONS[cfg.selection]
    sel_kw = {"tournament": {"k": cfg.tournament_k}, "rank": {"s": cfg.rank_s}, "roulette": {}}[cfg.selection]
    cross = CROSSOVERS[cfg.crossover]
    cx_kw = {"sbx": {"eta": cfg.sbx_eta}, "blx": {"alpha": cfg.blx_alpha}, "arith": {}}[cfg.crossover]

    pop = init_population(rng, mu, d, low, high)
    sig = np.full(mu, cfg.sigma0 * (high - low))
    fit = obj(pop)
    best_i = np.argmin(fit)
    best_x, best_f = pop[best_i].copy(), fit[best_i]
    hist = [(obj.evals, best_f, fit.mean(), np.median(sig))]

    while obj.evals + n_off <= cfg.max_evals:
        # 1. селекция
        par = select(rng, fit, 2 * n_pairs, **sel_kw)
        i1, i2 = par[:n_pairs], par[n_pairs:]
        P1, P2 = pop[i1], pop[i2]
        # 2. кроссовер (пары, не прошедшие по p_c, копируются без изменений)
        C1, C2 = cross(rng, P1, P2, **cx_kw)
        do_cx = (rng.random(n_pairs) < cfg.pc)[:, None]
        C1, C2 = np.where(do_cx, C1, P1), np.where(do_cx, C2, P2)
        child = np.vstack([C1, C2])[:n_off]
        # 3. мутация (σ потомка — среднее геометрическое σ родителей)
        if cfg.adaptive:
            s_par = np.sqrt(sig[i1] * sig[i2])
            s_child = self_adapt_sigma(rng, np.concatenate([s_par, s_par])[:n_off], tau)
        else:
            s_child = np.full(n_off, cfg.sigma0 * (high - low))
        child = mutate_gaussian(rng, child, s_child, cfg.pm)
        # 4. обработка границ и оценка
        child = repair_bounds(child, low, high, cfg.bounds)
        f_child = obj(child)
        # 5. элитизм: e лучших родителей переходят в следующее поколение
        elite_idx = np.argsort(fit)[:e]
        pop = np.vstack([pop[elite_idx], child])
        fit = np.concatenate([fit[elite_idx], f_child])
        sig = np.concatenate([sig[elite_idx], s_child])
        i = np.argmin(fit)
        if fit[i] < best_f:
            best_x, best_f = pop[i].copy(), fit[i]
        hist.append((obj.evals, best_f, fit.mean(), np.median(sig)))

    history = pd.DataFrame(hist, columns=["evals", "best", "mean", "sigma"])
    return {"best_x": best_x, "best_f": best_f, "evals": obj.evals, "history": history}

### 1.7. Базовые методы для сравнения

* **Случайный поиск** — равномерная выборка из $D$ с тем же бюджетом вычислений $f$.
* **Локальный поиск (1+1) с фиксированным шагом** — классический «hill climbing»: из текущей точки делается
  гауссовский шаг фиксированного размера, точка принимается, если она не хуже. Нужен для иллюстрации того,
  почему эта функция трудна для простого локального поиска.

История записывается в тех же точках по числу вычислений, что и у ГА (`checkpoints`), чтобы траектории
можно было рисовать на одной оси.

In [8]:
def run_random_search(max_evals, seed, checkpoints, func=sum_diff_powers, d=D, low=LOW, high=HIGH):
    rng = np.random.default_rng(seed)
    obj = Objective(func, d, low, high)
    fvals = obj(rng.uniform(low, high, size=(max_evals, d)))
    best_curve = np.minimum.accumulate(fvals)
    i = np.argmin(fvals)
    history = pd.DataFrame({"evals": checkpoints, "best": best_curve[np.asarray(checkpoints) - 1]})
    return {"best_f": fvals[i], "evals": obj.evals, "history": history}


def run_hill_climb(max_evals, seed, checkpoints, step=0.1, func=sum_diff_powers, d=D, low=LOW, high=HIGH):
    rng = np.random.default_rng(seed)
    obj = Objective(func, d, low, high)
    x = rng.uniform(low, high, d)
    fx = obj(x)[0]
    curve = np.empty(max_evals)
    curve[0] = fx
    for t in range(1, max_evals):
        y = repair_bounds(x + rng.normal(0, step * (high - low), d), low, high)
        fy = obj(y)[0]
        if fy <= fx:
            x, fx = y, fy
        curve[t] = fx
    history = pd.DataFrame({"evals": checkpoints, "best": curve[np.asarray(checkpoints) - 1]})
    return {"best_f": fx, "evals": obj.evals, "history": history}

### 1.8. Серии запусков, статистика и графики

* `run_series` — серия независимых запусков с seed `base_seed + r`, `r = 0..n_runs-1`.
* `summarize` — лучшее, среднее, медиана, стандартное отклонение, худшее, доля «успешных» запусков
  ($f < 10^{-6}$), среднее время.
* `plot_convergence` — медиана лучшего значения по серии, межквартильный диапазон и min–max (логарифмическая шкала).

In [9]:
SUCCESS_EPS = 1e-6


def run_series(runner, n_runs=30, base_seed=1000):
    """runner(seed) -> dict. Возвращает таблицу итогов и список историй."""
    rows, histories = [], []
    for r in range(n_runs):
        seed = base_seed + r
        t0 = time.perf_counter()
        res = runner(seed)
        rows.append({"seed": seed, "best_f": res["best_f"], "evals": res["evals"],
                     "time_s": time.perf_counter() - t0})
        histories.append(res["history"])
    return pd.DataFrame(rows), histories


def summarize(runs: pd.DataFrame, name=""):
    f = runs["best_f"]
    return pd.Series({"best": f.min(), "mean": f.mean(), "median": f.median(), "std": f.std(),
                      "worst": f.max(), f"success(f<{SUCCESS_EPS:g})": (f < SUCCESS_EPS).mean(),
                      "evals": runs["evals"].mean(), "time_s": runs["time_s"].mean()}, name=name)


def curves_matrix(histories, col="best"):
    """Матрица (runs, generations) значений колонки col и общая ось evals."""
    return np.vstack([h[col].to_numpy() for h in histories]), histories[0]["evals"].to_numpy()


def plot_convergence(series: dict, title, ax=None, band="iqr", show_minmax=False):
    """series: {label: histories}. Рисует медиану и диапазон по запускам."""
    if ax is None:
        _, ax = plt.subplots()
    for label, histories in series.items():
        M, x = curves_matrix(histories)
        M = np.maximum(M, 1e-300)                       # для лог-шкалы
        line, = ax.plot(x, np.median(M, axis=0), label=f"{label} (медиана)")
        if band == "iqr":
            ax.fill_between(x, np.quantile(M, 0.25, axis=0), np.quantile(M, 0.75, axis=0),
                            color=line.get_color(), alpha=0.25)
        if show_minmax:
            ax.plot(x, M.min(axis=0), color=line.get_color(), ls=":", lw=1)
            ax.plot(x, M.max(axis=0), color=line.get_color(), ls="--", lw=1)
    ax.set_yscale("log")
    ax.set_xlabel("число вычислений f")
    ax.set_ylabel("лучшее найденное f (log)")
    ax.set_title(title)
    ax.legend()
    return ax


def mann_whitney(a, b):
    """Двусторонний U-тест Манна—Уитни для итоговых значений двух серий."""
    return stats.mannwhitneyu(a, b, alternative="two-sided").pvalue

### 1.9. Проверка корректности функций

Короткие проверки: минимум функции в нуле, известные значения, отражение не выводит за границы,
операторы сохраняют размерности, один запуск ГА воспроизводим при фиксированном seed.

In [10]:
assert sum_diff_powers(np.zeros(D))[0] == 0.0
assert np.isclose(sum_diff_powers(np.ones(D))[0], D)              # |1|^p = 1 для каждой координаты
assert np.isclose(sum_diff_powers(np.array([0.5, 0.5]))[0], 0.25 + 0.125)

_rng = np.random.default_rng(0)
_X = _rng.uniform(-5, 5, size=(1000, D))
_R = repair_bounds(_X, LOW, HIGH)
assert _R.min() >= LOW and _R.max() <= HIGH
assert np.allclose(repair_bounds(np.array([[1.2, -1.3, 0.4]]), LOW, HIGH), [[0.8, -0.7, 0.4]])

_fit = sum_diff_powers(_R)
for name, sel in SELECTIONS.items():
    picked = sel(_rng, _fit, 5000)
    # селекция должна в среднем выбирать особей лучше среднего по популяции
    assert _fit[picked].mean() < _fit.mean(), name
for name, cx in CROSSOVERS.items():
    c1, c2 = cx(_rng, _R[:10], _R[10:20])
    assert c1.shape == c2.shape == (10, D), name

_a = run_ga(GAConfig(max_evals=2000), seed=42)
_b = run_ga(GAConfig(max_evals=2000), seed=42)
assert _a["best_f"] == _b["best_f"] and _a["evals"] <= 2000
print("Все проверки пройдены. Пробный запуск ГА (2000 вычислений): f =", _a["best_f"])

Все проверки пройдены. Пробный запуск ГА (2000 вычислений): f = 3.881457584054371e-05
